In [1]:
!nvidia-smi

'nvidia-smi' is not recognized as an internal or external command,
operable program or batch file.


In [4]:
!pip install -q pypdf python-dotenv

In [6]:
!pip -q install git+https://github.com/huggingface/transformers

In [7]:
!pip install -q datasets loralib sentencepiece

In [9]:
!pip install -q einops accelerate langchain bitsandbytes

In [11]:
!pip install sentence_transformers

  Using cached sentence_transformers-2.7.0-py3-none-any.whl.metadata (11 kB)
Using cached sentence_transformers-2.7.0-py3-none-any.whl (171 kB)


In [12]:
!pip install llama-index

   ---------------------------------------- 0.0/15.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/15.4 MB 991.0 kB/s eta 0:00:16
   ---------------------------------------- 0.1/15.4 MB 656.4 kB/s eta 0:00:24
   ---------------------------------------- 0.1/15.4 MB 950.9 kB/s eta 0:00:17
   ---------------------------------------- 0.2/15.4 MB 930.9 kB/s eta 0:00:17
    --------------------------------------- 0.3/15.4 MB 1.1 MB/s eta 0:00:14
    --------------------------------------- 0.3/15.4 MB 1.2 MB/s eta 0:00:13
   - -------------------------------------- 0.4/15.4 MB 1.2 MB/s eta 0:00:13
   - -------------------------------------- 0.5/15.4 MB 1.3 MB/s eta 0:00:12
   - -------------------------------------- 0.5/15.4 MB 1.2 MB/s eta 0:00:13
   - -------------------------------------- 0.6/15.4 MB 1.3 MB/s eta 0:00:12
   - -------------------------------------- 0.6/15.4 MB 1.3 MB/s eta 0:00:12
   - -------------------------------------- 0.8/15.4 MB 1.3 MB/s eta 0:00:1

In [13]:
!pip install -q chromadb

In [ ]:
!pip install llama-index-llms-huggingface

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 330.1/330.1 kB 8.0 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface-hub 0.22.2
    Uninstalling huggingface-hub-0.22.2:
      Successfully uninstalled huggingface-hub-0.22.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
datasets 2.19.0 requires huggingface-hub>=0.21.2, but you have huggingface-hub 0.20.3 which is incompatible.


In [ ]:
!pip install llama-index-vector-stores-chroma
!pip install llama-index

In [ ]:
from llama_index.core import SimpleDirectoryReader, ServiceContext, StorageContext, VectorStoreIndex

from llama_index.llms.huggingface import HuggingFaceLLM

#from llama_index.embeddings import HuggingFaceEmbedding
from langchain.embeddings import HuggingFaceEmbeddings


from llama_index.vector_stores.chroma import ChromaVectorStore

import chromadb

from IPython.display import Markdown, display

In [ ]:
chroma_client = chromadb.PersistentClient()
chroma_collection = chroma_client.create_collection('climate_report')
vector_store = ChromaVectorStore(chroma_collection=chroma_collection)
storage_context = StorageContext.from_defaults(vector_store=vector_store)

In [ ]:
documents = SimpleDirectoryReader("data").load_data()

In [ ]:
#from llama_index.prompts.prompts import SimpleInputPrompt
from llama_index.core.prompts.prompts import SimpleInputPrompt

system_prompt = "You are a Q&A assistant. Your goal is to answer questions as accurately as possible based on the instructions and context provided."
# This will wrap the default prompts that are internal to llama-index
query_wrapper_prompt = "<|USER|>{query_str}<|ASSISTANT|>"

In [ ]:
!huggingface-cli login


    _|    _|  _|    _|    _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|_|_|_|    _|_|      _|_|_|  _|_|_|_|
    _|    _|  _|    _|  _|        _|          _|    _|_|    _|  _|            _|        _|    _|  _|        _|
    _|_|_|_|  _|    _|  _|  _|_|  _|  _|_|    _|    _|  _|  _|  _|  _|_|      _|_|_|    _|_|_|_|  _|        _|_|_|
    _|    _|  _|    _|  _|    _|  _|    _|    _|    _|    _|_|  _|    _|      _|        _|    _|  _|        _|
    _|    _|    _|_|      _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|        _|    _|    _|_|_|  _|_|_|_|

    To login, `huggingface_hub` requires a token generated from https://huggingface.co/settings/tokens .
Token: 
Add token as git credential? (Y/n) n
Token is valid (permission: read).
Your token has been saved to /root/.cache/huggingface/token
Login successful


In [ ]:
import torch
torch.set_default_device('cuda')

In [26]:
llm = HuggingFaceLLM(
    context_window=8000,
    max_new_tokens=256,
    generate_kwargs={"temperature": 0.1, "do_sample": True},
    system_prompt=system_prompt,
    query_wrapper_prompt=query_wrapper_prompt,
    tokenizer_name="mistralai/Mistral-7B-Instruct-v0.1",
    model_name="mistralai/Mistral-7B-Instruct-v0.1",
    device_map="auto",
    tokenizer_kwargs={"max_length": 8000},
    model_kwargs={"torch_dtype": torch.float16}

)

OutOfMemoryError: CUDA out of memory. Tried to allocate 250.00 MiB. GPU 0 has a total capacity of 14.75 GiB of which 23.06 MiB is free. Process 9303 has 14.72 GiB memory in use. Of the allocated memory 14.47 GiB is allocated by PyTorch, and 139.46 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)